In [2]:
using DifferentialEquations
using LinearAlgebra
using Random
using Statistics
using Plots
using LaTeXStrings
using Dates
using Printf

# Parameters
const E_PARAM = 0.25
const D_PARAM = 0.5

# Simulation parameters
const MATRIX_SIZE = 10
const P_VALS = range(0, 1, length=11)
const Q_VALS = range(0, 1, length=11)
const NUM_TRIALS = 100

# ODE solver parameters
const TIME_SPAN = (0.0, 600.0)
const CONVERGENCE_WINDOW = 25
const CONVERGENCE_TOL = 0.01

function generate_weight_matrix(matrix_size::Int, p::Float64, q::Float64)
    prob_1 = 1 - p
    prob_2 = prob_1 + p * (1 - q) / 2
    prob_3 = prob_2 + p * (1 - q) / 2
    
    w_excited = -1 + E_PARAM
    w_inhibited = -1 - D_PARAM
    
    mat = zeros(matrix_size, matrix_size)
    
    for i in 1:(matrix_size-1)
        for j in (i+1):matrix_size
            r = rand()
            
            if r < prob_1
                mat[i, j] = w_inhibited
                mat[j, i] = w_inhibited
            elseif r < prob_2
                mat[i, j] = w_inhibited
                mat[j, i] = w_excited
            elseif r < prob_3
                mat[i, j] = w_excited
                mat[j, i] = w_inhibited
            else
                mat[i, j] = w_excited
                mat[j, i] = w_excited
            end
        end
    end
    
    return mat
end

function system_ode!(dx, x, W, t)
    mul!(dx, W, x)
    @inbounds for i in eachindex(dx)
        dx[i] = -x[i] + max(0.0, dx[i] + 1.0)
    end
end

function check_convergence(sol)
    if length(sol.t) < CONVERGENCE_WINDOW
        return false
    end
    
    final_state = sol.u[end]
    recent_states = sol.u[(end-CONVERGENCE_WINDOW+1):end]
    
    for state in recent_states
        for i in eachindex(state)
            if abs(final_state[i] - state[i]) > CONVERGENCE_TOL
                return false
            end
        end
    end
    return true
end

function run_simulation()
    num_p = length(P_VALS)
    num_q = length(Q_VALS)
    convergence_heatmap = zeros(num_q, num_p)
    
    # Store full solutions instead of just final states
    all_solutions = Dict()
    all_weight_matrices = Dict()
    plot_solutions = Matrix{Any}(undef, num_q, num_p)
    
    println("Starting simulation...")
    
    for (p_idx, p) in enumerate(P_VALS)
        for (q_idx, q) in enumerate(Q_VALS)
            trial_convergence_results = Float64[]
            
            for trial_num in 1:NUM_TRIALS
                W = generate_weight_matrix(MATRIX_SIZE, p, q)
                x0 = rand(MATRIX_SIZE)
                
                prob = ODEProblem((dx, x, p, t) -> system_ode!(dx, x, W, t), 
                                 x0, TIME_SPAN)
                
                sol = solve(prob, Tsit5(), saveat=1.0, reltol=1e-6, abstol=1e-8)
                
                # Store solutions
                key = (MATRIX_SIZE, p_idx, q_idx)
                if !haskey(all_solutions, key)
                    all_solutions[key] = []
                    all_weight_matrices[key] = []
                end
                push!(all_solutions[key], sol)
                push!(all_weight_matrices[key], copy(W))
                
                is_converged = check_convergence(sol)
                push!(trial_convergence_results, is_converged ? 1.0 : 0.0)
                
                if trial_num == 1
                    plot_solutions[q_idx, p_idx] = sol
                end
            end
            
            convergence_heatmap[q_idx, p_idx] = mean(trial_convergence_results)
            println("p=$(round(p, digits=1)), q=$(round(q, digits=1)) | " *
                   "Mean Convergence: $(round(mean(trial_convergence_results), digits=2))")
        end
    end
    
    println("\nSimulation complete. Generating plots...")
    
    # Create trajectory plots
    plots_array = []
    for q_idx in 1:num_q
        for p_idx in 1:num_p
            sol = plot_solutions[num_q + 1 - q_idx, p_idx]
            p_val = P_VALS[p_idx]
            q_val = Q_VALS[num_q + 1 - q_idx]
            
            plt = plot(sol.t, reduce(hcat, sol.u)', 
                      legend=false, linewidth=0.5,
                      title="p=$(round(p_val, digits=1)), q=$(round(q_val, digits=1))",
                      titlefontsize=8,
                      tickfontsize=6)
            
            if q_idx == 1
                xlabel!(plt, "Time t")
            end
            if p_idx == 1
                ylabel!(plt, "State x_i")
            end
            
            push!(plots_array, plt)
        end
    end
    
    fig_trajectories = plot(plots_array..., layout=(num_q, num_p),
                           size=(1500, 1500),
                           plot_title="System Trajectories for Matrix Size $MATRIX_SIZE")
    
    savefig(fig_trajectories, "trajectories.pdf")
    
    return convergence_heatmap, all_solutions, all_weight_matrices
end



run_simulation (generic function with 1 method)

In [3]:
# Folder creation
function create_analysis_folder(matrix_size::Int)
    timestamp = Dates.format(now(), "yyyymmdd_HHMMSS")
    run_folder = "analysis_run_$(timestamp)_N$(matrix_size)"
    mkpath(run_folder)
    println("Created analysis folder: $run_folder")
    println("All figures will be saved to this folder.\n")
    return run_folder
end

# BLOCK 1: ACTIVE NEURON ANALYSIS
function analyze_active_neurons(all_solutions, P_VALS, Q_VALS, 
                                 matrix_size::Int, run_folder::String)
    println("="^70)
    println("BLOCK 1: ACTIVE NEURON ANALYSIS")
    println("="^70)
    
    ACTIVE_THRESHOLD = 0.05
    
    num_p = length(P_VALS)
    num_q = length(Q_VALS)
    
    active_mean_heatmap = zeros(num_q, num_p)
    active_std_heatmap = zeros(num_q, num_p)
    active_min_heatmap = zeros(num_q, num_p)
    active_max_heatmap = zeros(num_q, num_p)
    
    println("Analyzing active neurons (threshold: x > $ACTIVE_THRESHOLD)...\n")
    
    for p_idx in 1:num_p
        for q_idx in 1:num_q
            key = (matrix_size, p_idx, q_idx)
            
            if !haskey(all_solutions, key)
                println("Warning: No data for p_idx=$p_idx, q_idx=$q_idx")
                continue
            end
            
            trial_active_counts = Float64[]
            for sol in all_solutions[key]
                final_state = sol.u[end]
                num_active = sum(final_state .> ACTIVE_THRESHOLD)
                push!(trial_active_counts, num_active)
            end
            
            active_mean_heatmap[q_idx, p_idx] = mean(trial_active_counts)
            active_std_heatmap[q_idx, p_idx] = std(trial_active_counts)
            active_min_heatmap[q_idx, p_idx] = minimum(trial_active_counts)
            active_max_heatmap[q_idx, p_idx] = maximum(trial_active_counts)
            
            p_val = P_VALS[p_idx]
            q_val = Q_VALS[q_idx]
            @printf("p=%.1f, q=%.1f | Active: %.1f±%.1f [%.0f, %.0f]\n",
                    p_val, q_val, 
                    mean(trial_active_counts), std(trial_active_counts),
                    minimum(trial_active_counts), maximum(trial_active_counts))
        end
    end
    
    # Visualization setup
    p_vals_vec = collect(P_VALS)
    q_vals_vec = collect(Q_VALS)
    
    expected_max_active = max(matrix_size * 0.8, 3 * log(matrix_size))
    actual_max = maximum(active_mean_heatmap)
    vmax_count = min(matrix_size, max(expected_max_active, actual_max * 1.1))
    vmax_std = vmax_count / 4
    
    println("\nColor scale info:")
    @printf("  Expected max active (log scale): %.1f\n", expected_max_active)
    @printf("  Actual max observed: %.1f\n", actual_max)
    @printf("  Using vmax: %.1f\n", vmax_count)
    
    # Figure 1: Four-panel statistics
    p1 = heatmap(p_vals_vec, q_vals_vec, active_mean_heatmap,
                 title="Mean Active Neurons",
                 xlabel="p (interaction probability)",
                 ylabel="q (excitatory probability)",
                 color=:viridis, clims=(0, vmax_count),
                 titlefontsize=10, labelfontsize=9,
                 aspect_ratio=:auto)
    
    p2 = heatmap(p_vals_vec, q_vals_vec, active_std_heatmap,
                 title="Std Dev of Active Neurons",
                 xlabel="p (interaction probability)",
                 ylabel="q (excitatory probability)",
                 color=:plasma, clims=(0, vmax_std),
                 titlefontsize=10, labelfontsize=9,
                 aspect_ratio=:auto)
    
    p3 = heatmap(p_vals_vec, q_vals_vec, active_min_heatmap,
                 title="Minimum Active Neurons",
                 xlabel="p (interaction probability)",
                 ylabel="q (excitatory probability)",
                 color=:cividis, clims=(0, vmax_count),
                 titlefontsize=10, labelfontsize=9,
                 aspect_ratio=:auto)
    
    p4 = heatmap(p_vals_vec, q_vals_vec, active_max_heatmap,
                 title="Maximum Active Neurons",
                 xlabel="p (interaction probability)",
                 ylabel="q (excitatory probability)",
                 color=:inferno, clims=(0, vmax_count),
                 titlefontsize=10, labelfontsize=9,
                 aspect_ratio=:auto)
    
    fig_stats = plot(p1, p2, p3, p4, layout=(2,2), size=(1200, 1200),
                     plot_title="Active Neuron Statistics (N=$matrix_size)")
    
    savefig(fig_stats, joinpath(run_folder, "active_neurons_statistics.pdf"))
    savefig(fig_stats, joinpath(run_folder, "active_neurons_statistics.png"))
    
    # Figure 2: Fraction of active neurons
    active_fraction_heatmap = active_mean_heatmap ./ matrix_size
    
    fig_frac = heatmap(p_vals_vec, q_vals_vec, active_fraction_heatmap,
                       title="Fraction of Active Neurons (N=$matrix_size)",
                       xlabel="p (interaction probability)",
                       ylabel="q (excitatory probability)",
                       color=:RdYlGn_11, clims=(0, 1),
                       size=(800, 700),
                       titlefontsize=12, labelfontsize=10)
    
    savefig(fig_frac, joinpath(run_folder, "active_neurons_fraction.pdf"))
    savefig(fig_frac, joinpath(run_folder, "active_neurons_fraction.png"))
    
    # Summary
    println("\n" * "="^70)
    println("BLOCK 1 SUMMARY: ACTIVE NEURONS")
    println("="^70)
    @printf("Threshold: x > %.2f\n", ACTIVE_THRESHOLD)
    @printf("Mean active neurons: %.2f ± %.2f\n", 
            mean(active_mean_heatmap), std(active_mean_heatmap))
    @printf("Mean fraction active: %.3f ± %.3f\n",
            mean(active_fraction_heatmap), std(active_fraction_heatmap))
    @printf("Range: [%.1f, %.1f] neurons\n",
            minimum(active_mean_heatmap), maximum(active_mean_heatmap))
    @printf("Max variability (std): %.2f\n", maximum(active_std_heatmap))
    println("\nFigures saved to: $run_folder/")
    println("  - active_neurons_statistics.pdf/png")
    println("  - active_neurons_fraction.pdf/png")
    println("="^70 * "\n")
    
    return active_mean_heatmap, active_fraction_heatmap
end

# BLOCK 2: FIXED POINT VS NONLINEAR ANALYSIS
function analyze_fixedpt_vs_nonlinear(all_solutions, P_VALS, Q_VALS, 
                                       matrix_size::Int, run_folder::String)
    println("="^70)
    println("BLOCK 2: FIXED POINT VS NONLINEAR ANALYSIS")
    println("="^70)
    
    ACTIVE_THRESHOLD = 0.05
    DERIVATIVE_WINDOW = 15  # Number of time points to look back
    DERIVATIVE_THRESHOLD = 0.01  # Max change per time step to be "stable"
    
    num_p = length(P_VALS)
    num_q = length(Q_VALS)
    
    fixedpt_mean_heatmap = zeros(num_q, num_p)
    fixedpt_std_heatmap = zeros(num_q, num_p)
    nonlinear_mean_heatmap = zeros(num_q, num_p)
    nonlinear_std_heatmap = zeros(num_q, num_p)
    
    println("Analyzing fixed point vs nonlinear dynamics...")
    @printf("  Active threshold: x > %.2f\n", ACTIVE_THRESHOLD)
    @printf("  Derivative window: last %d time points\n", DERIVATIVE_WINDOW)
    @printf("  Derivative threshold: %.2f (max change per time step)\n\n", DERIVATIVE_THRESHOLD)
    
    for p_idx in 1:num_p
        for q_idx in 1:num_q
            key = (matrix_size, p_idx, q_idx)
            
            if !haskey(all_solutions, key)
                println("Warning: No data for p_idx=$p_idx, q_idx=$q_idx")
                continue
            end
            
            trial_fixedpt_counts = Float64[]
            trial_nonlinear_counts = Float64[]
            
            for sol in all_solutions[key]
                # Get the last DERIVATIVE_WINDOW time points
                if length(sol.t) < DERIVATIVE_WINDOW
                    println("Warning: Solution too short, skipping")
                    continue
                end
                
                recent_states = sol.u[(end-DERIVATIVE_WINDOW+1):end]
                final_state = sol.u[end]
                
                # Only consider active neurons
                is_active = final_state .> ACTIVE_THRESHOLD
                
                # Calculate max change over the window for each neuron
                max_changes = zeros(matrix_size)
                for i in 1:matrix_size
                    if is_active[i]
                        # Extract time series for this neuron
                        neuron_values = [state[i] for state in recent_states]
                        # Calculate maximum absolute change between consecutive time points
                        changes = abs.(diff(neuron_values))
                        max_changes[i] = maximum(changes)
                    end
                end
                
                # Classify based on dynamics
                at_fixed_point = (max_changes .< DERIVATIVE_THRESHOLD) .& is_active
                nonlinear_dynamics = (max_changes .>= DERIVATIVE_THRESHOLD) .& is_active
                
                num_fixedpt = sum(at_fixed_point)
                num_nonlinear = sum(nonlinear_dynamics)
                
                push!(trial_fixedpt_counts, num_fixedpt)
                push!(trial_nonlinear_counts, num_nonlinear)
            end
            
            fixedpt_mean_heatmap[q_idx, p_idx] = mean(trial_fixedpt_counts)
            fixedpt_std_heatmap[q_idx, p_idx] = std(trial_fixedpt_counts)
            nonlinear_mean_heatmap[q_idx, p_idx] = mean(trial_nonlinear_counts)
            nonlinear_std_heatmap[q_idx, p_idx] = std(trial_nonlinear_counts)
            
            p_val = P_VALS[p_idx]
            q_val = Q_VALS[q_idx]
            @printf("p=%.1f, q=%.1f | FixedPt: %.1f±%.1f | Nonlinear: %.1f±%.1f\n",
                    p_val, q_val,
                    mean(trial_fixedpt_counts), std(trial_fixedpt_counts),
                    mean(trial_nonlinear_counts), std(trial_nonlinear_counts))
        end
    end
    
    # Visualization
    p_vals_vec = collect(P_VALS)
    q_vals_vec = collect(Q_VALS)
    
    expected_max_active = max(matrix_size * 0.8, 3 * log(matrix_size))
    actual_max = max(maximum(fixedpt_mean_heatmap), maximum(nonlinear_mean_heatmap))
    vmax_count = min(matrix_size, max(expected_max_active, actual_max * 1.1))
    vmax_std = vmax_count / 4
    
    # Figure 1: Mean counts comparison
    p1 = heatmap(p_vals_vec, q_vals_vec, fixedpt_mean_heatmap,
                 title="Fixed Point Neurons (stable)",
                 xlabel="p (interaction probability)",
                 ylabel="q (excitatory probability)",
                 color=:cividis, clims=(0, vmax_count),
                 titlefontsize=9, labelfontsize=9)
    
    p2 = heatmap(p_vals_vec, q_vals_vec, nonlinear_mean_heatmap,
                 title="Nonlinear Dynamics (oscillating/changing)",
                 xlabel="p (interaction probability)",
                 ylabel="q (excitatory probability)",
                 color=:plasma, clims=(0, vmax_count),
                 titlefontsize=9, labelfontsize=9)
    
    fig_means = plot(p1, p2, layout=(1,2), size=(1200, 550),
                     plot_title="Fixed Point vs Nonlinear Dynamics (N=$matrix_size)")
    
    savefig(fig_means, joinpath(run_folder, "fixedpt_vs_nonlinear_means.pdf"))
    savefig(fig_means, joinpath(run_folder, "fixedpt_vs_nonlinear_means.png"))
    
    # Figure 2: Standard deviations
    p1s = heatmap(p_vals_vec, q_vals_vec, fixedpt_std_heatmap,
                  title="Fixed Point Neurons",
                  xlabel="p (interaction probability)",
                  ylabel="q (excitatory probability)",
                  color=:cividis, clims=(0, vmax_std),
                  titlefontsize=9, labelfontsize=9)
    
    p2s = heatmap(p_vals_vec, q_vals_vec, nonlinear_std_heatmap,
                  title="Nonlinear Dynamics",
                  xlabel="p (interaction probability)",
                  ylabel="q (excitatory probability)",
                  color=:plasma, clims=(0, vmax_std),
                  titlefontsize=9, labelfontsize=9)
    
    fig_stds = plot(p1s, p2s, layout=(1,2), size=(1200, 550),
                    plot_title="Variability: Fixed Point vs Nonlinear (N=$matrix_size)")
    
    savefig(fig_stds, joinpath(run_folder, "fixedpt_vs_nonlinear_stds.pdf"))
    savefig(fig_stds, joinpath(run_folder, "fixedpt_vs_nonlinear_stds.png"))
    
    # Figure 3: Fractions
    fixedpt_frac = fixedpt_mean_heatmap ./ matrix_size
    nonlinear_frac = nonlinear_mean_heatmap ./ matrix_size
    
    p1f = heatmap(p_vals_vec, q_vals_vec, fixedpt_frac,
                  title="Fixed Point Neurons",
                  xlabel="p (interaction probability)",
                  ylabel="q (excitatory probability)",
                  color=:BuGn_9, clims=(0, 1),
                  titlefontsize=9, labelfontsize=9)
    
    p2f = heatmap(p_vals_vec, q_vals_vec, nonlinear_frac,
                  title="Nonlinear Dynamics",
                  xlabel="p (interaction probability)",
                  ylabel="q (excitatory probability)",
                  color=:RdYlGn_11, clims=(0, 1),
                  titlefontsize=9, labelfontsize=9)
    
    fig_fracs = plot(p1f, p2f, layout=(1,2), size=(1200, 550),
                     plot_title="Fraction: Fixed Point vs Nonlinear (N=$matrix_size)")
    
    savefig(fig_fracs, joinpath(run_folder, "fixedpt_vs_nonlinear_fractions.pdf"))
    savefig(fig_fracs, joinpath(run_folder, "fixedpt_vs_nonlinear_fractions.png"))
    
    # Figure 4: Difference and ratio analysis
    difference_heatmap = fixedpt_mean_heatmap .- nonlinear_mean_heatmap
    vmax_diff = maximum(abs.(difference_heatmap))
    
    p1d = heatmap(p_vals_vec, q_vals_vec, difference_heatmap,
                  title="Difference (Fixed Pt - Nonlinear)",
                  xlabel="p (interaction probability)",
                  ylabel="q (excitatory probability)",
                  color=:RdBu_11, clims=(-vmax_diff, vmax_diff),
                  titlefontsize=9, labelfontsize=9)
    
    denominator = fixedpt_mean_heatmap .+ nonlinear_mean_heatmap
    ratio_heatmap = zeros(size(fixedpt_mean_heatmap))
    mask = denominator .> 0.1
    ratio_heatmap[mask] = fixedpt_mean_heatmap[mask] ./ denominator[mask]
    
    p2d = heatmap(p_vals_vec, q_vals_vec, ratio_heatmap,
                  title="Ratio: Fixed Pt / (Fixed Pt + Nonlinear)",
                  xlabel="p (interaction probability)",
                  ylabel="q (excitatory probability)",
                  color=:PuOr_11, clims=(0, 1),
                  titlefontsize=9, labelfontsize=9)
    
    fig_diff = plot(p1d, p2d, layout=(1,2), size=(1200, 550),
                    plot_title="Comparative Analysis (N=$matrix_size)")
    
    savefig(fig_diff, joinpath(run_folder, "fixedpt_vs_nonlinear_comparison.pdf"))
    savefig(fig_diff, joinpath(run_folder, "fixedpt_vs_nonlinear_comparison.png"))
    
    # Summary
    println("\n" * "="^70)
    println("BLOCK 2 SUMMARY: FIXED POINT VS NONLINEAR DYNAMICS")
    println("="^70)
    @printf("Active Threshold: x > %.2f\n", ACTIVE_THRESHOLD)
    @printf("Derivative Window: last %d time points\n", DERIVATIVE_WINDOW)
    @printf("Derivative Threshold: %.2f\n\n", DERIVATIVE_THRESHOLD)
    
    println("FIXED POINT NEURONS (active, stable):")
    @printf("  Mean: %.2f ± %.2f\n", mean(fixedpt_mean_heatmap), std(fixedpt_mean_heatmap))
    @printf("  Fraction: %.3f ± %.3f\n", mean(fixedpt_frac), std(fixedpt_frac))
    @printf("  Range: [%.1f, %.1f]\n\n", minimum(fixedpt_mean_heatmap), maximum(fixedpt_mean_heatmap))
    
    println("NONLINEAR DYNAMICS (active, oscillating/changing):")
    @printf("  Mean: %.2f ± %.2f\n", mean(nonlinear_mean_heatmap), std(nonlinear_mean_heatmap))
    @printf("  Fraction: %.3f ± %.3f\n", mean(nonlinear_frac), std(nonlinear_frac))
    @printf("  Range: [%.1f, %.1f]\n\n", minimum(nonlinear_mean_heatmap), maximum(nonlinear_mean_heatmap))
    
    println("COMPARATIVE METRICS:")
    @printf("  Mean difference (Fixed Pt - Nonlinear): %.2f\n", mean(difference_heatmap))
    @printf("  Mean ratio (Fixed Pt / Total): %.3f\n", mean(ratio_heatmap[mask]))
    println("\nFigures saved to: $run_folder/")
    println("  - fixedpt_vs_nonlinear_means.pdf/png")
    println("  - fixedpt_vs_nonlinear_stds.pdf/png")
    println("  - fixedpt_vs_nonlinear_fractions.pdf/png")
    println("  - fixedpt_vs_nonlinear_comparison.pdf/png")
    println("="^70 * "\n")
    
    return fixedpt_mean_heatmap, nonlinear_mean_heatmap
end



analyze_fixedpt_vs_nonlinear (generic function with 1 method)

In [4]:
# MAIN EXECUTION
println("Running simulation (this will take a while)...")
@time convergence_heatmap, all_solutions, weight_matrices = run_simulation()
println("\nDone with simulation!")

# Create analysis folder
ANALYZE_MATRIX_SIZE = MATRIX_SIZE
run_folder = create_analysis_folder(ANALYZE_MATRIX_SIZE)

# Block 1
active_mean, active_frac = analyze_active_neurons(
    all_solutions, P_VALS, Q_VALS, ANALYZE_MATRIX_SIZE, run_folder
)

# Block 2
fixedpt_mean, nonlinear_mean = analyze_fixedpt_vs_nonlinear(
    all_solutions, P_VALS, Q_VALS, ANALYZE_MATRIX_SIZE, run_folder
)

println("Analysis complete!")

Running simulation (this will take a while)...
Starting simulation...
p=0.0, q=0.0 | Mean Convergence: 1.0
p=0.0, q=0.1 | Mean Convergence: 1.0
p=0.0, q=0.2 | Mean Convergence: 1.0
p=0.0, q=0.3 | Mean Convergence: 1.0
p=0.0, q=0.4 | Mean Convergence: 1.0
p=0.0, q=0.5 | Mean Convergence: 1.0
p=0.0, q=0.6 | Mean Convergence: 1.0
p=0.0, q=0.7 | Mean Convergence: 1.0
p=0.0, q=0.8 | Mean Convergence: 1.0
p=0.0, q=0.9 | Mean Convergence: 1.0
p=0.0, q=1.0 | Mean Convergence: 1.0
p=0.1, q=0.0 | Mean Convergence: 1.0
p=0.1, q=0.1 | Mean Convergence: 0.98
p=0.1, q=0.2 | Mean Convergence: 0.97
p=0.1, q=0.3 | Mean Convergence: 1.0
p=0.1, q=0.4 | Mean Convergence: 1.0
p=0.1, q=0.5 | Mean Convergence: 1.0
p=0.1, q=0.6 | Mean Convergence: 1.0
p=0.1, q=0.7 | Mean Convergence: 1.0
p=0.1, q=0.8 | Mean Convergence: 1.0
p=0.1, q=0.9 | Mean Convergence: 1.0
p=0.1, q=1.0 | Mean Convergence: 1.0
p=0.2, q=0.0 | Mean Convergence: 0.85
p=0.2, q=0.1 | Mean Convergence: 0.94
p=0.2, q=0.2 | Mean Convergence: 0.99
